# BDAPP · Predicting organoid age and remaining life
### Maral Farhatnejad · Sejong University · KINJEX · October 2026

**One self-contained Colab notebook for your final course project.**

**Research questions**
1. Can the preceding hour of spontaneous activity predict biological age in an unseen organoid?
2. Does activity predict remaining life **when death endpoints are independently verified**?
3. Do environmental measurements explain some of the apparent age/activity relationship?

**Start here**
1. Upload this `.ipynb` through **Colab → File → Upload notebook**.
2. Put the **extracted** `fs369data`, `fs437`, `22755004`, and optional DANDI folders in one private Google Drive folder.
3. In **Settings**, set `DATA_ROOT` and `OUTPUT_ROOT`. CPU runtime is sufficient for these models.
4. Add the real dates and channel assignments in `organoids.csv`. If absent, the notebook creates a blank template and tells you what to fill. Missing units/dates are never invented.
5. Choose **Runtime → Run all**. Open `BDAPP_REPORT.html` and download the private results ZIP at the end.

**What is already implemented:** file inspection; chunked CSV/TSV/Parquet/HDF5 event ingestion; resumable checkpoints; verified-exposure windows; artifact exclusion; neural features; fold-isolated preprocessing; leave-one-organoid-out evaluation; FS369 → FS437 transfer; dummy/rate/linear/boosting/clock/shuffle controls; feature ablations; held-out permutation importance; optional SHAP; conditional environmental/survival analyses; native TU Munich MAT and NWB Units descriptive readers; figures and an HTML report.

**Validation boundary:** the real FS369/FS437 archives were not available while this notebook was built. The main pipeline is tested on explicitly synthetic event fixtures, and the public MAT reader is tested on actual files from Zenodo 22755004. Unrecognized private export layouts need a source mapping or a small adapter; this is not a claim of verification on your 300 GB archive.

**Private research.** Nothing is sent to an AI service or published. Drive mounting uses Google's normal authorization. Keep recordings, derived tables, figures and saved notebook outputs private according to your specific FinalSpark agreement.

## 1 · Prepare the runtime
The cell installs missing scientific dependencies only. It records the versions used in each results folder. No GPU is required. Colab runtimes and Drive I/O are limited; avoid uploading 300 GB through the browser upload widget. Use Drive or your own local runtime. The pipeline reads event tables in chunks and never loads the full archive into RAM.

In [ ]:
import importlib.util, importlib.metadata, subprocess, sys, os
from packaging.version import Version
dependencies = {
    'numpy': ('numpy>=1.26,<3', 'numpy', '1.26'),
    'pandas': ('pandas>=2.2,<3', 'pandas', '2.2'),
    'scipy': ('scipy>=1.11,<2', 'scipy', '1.11'),
    'sklearn': ('scikit-learn>=1.5,<2', 'scikit-learn', '1.5'),
    'h5py': ('h5py>=3.11,<4', 'h5py', '3.11'),
    'pyarrow': ('pyarrow>=16,<25', 'pyarrow', '16'),
    'duckdb': ('duckdb>=1.1,<2', 'duckdb', '1.1'),
    'matplotlib': ('matplotlib>=3.8,<4', 'matplotlib', '3.8'),
    'seaborn': ('seaborn>=0.13,<1', 'seaborn', '0.13'),
    'statsmodels': ('statsmodels>=0.14,<1', 'statsmodels', '0.14'),
    'lifelines': ('lifelines>=0.30,<1', 'lifelines', '0.30'),
}
needed=[]
for module,(spec,distribution,minimum) in dependencies.items():
    try:
        ready=importlib.util.find_spec(module) is not None and Version(importlib.metadata.version(distribution))>=Version(minimum)
    except Exception:
        ready=False
    if not ready: needed.append(spec)
if needed:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*needed])
print('Scientific environment ready. If Colab asks for a runtime restart after an upgrade, restart and Run all again.')

## 2 · Settings — the main controls
Use `real` for course results. `demo` explicitly generates small synthetic recordings in a separate folder so you can inspect the workflow before supplying data. Demo figures and reports are labeled and must not be used as research findings.

`WINDOW_SECONDS=3600` matches your proposal. `MIN_COVERAGE=0.95` is the minimum verified exposure for an eligible window. Fully observed windows are required for timing/binned features, so gaps are not treated as silence.

In [ ]:
MODE = "real" # @param ["real", "demo"]
DATA_LOCATION = "Google Drive" # @param ["Google Drive", "Local / Colab files"]
DATA_ROOT = "/content/drive/MyDrive/KINJEX_BDAPP/data" # @param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/KINJEX_BDAPP/results" # @param {type:"string"}
WINDOW_SECONDS = 3600 # @param {type:"integer"}
MIN_COVERAGE = 0.95 # @param {type:"number"}
CHUNK_ROWS = 200000 # @param {type:"integer"}
MAX_WINDOW_EVENTS = 2000000 # @param {type:"integer"}
SOURCE_TIMEZONE = "UTC" # @param {type:"string"}
RUN_ABLATIONS = True # @param {type:"boolean"}
RUN_SHAP = False # @param {type:"boolean"}
SEED = 42 # @param {type:"integer"}

if DATA_LOCATION == 'Google Drive':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('Outside hosted Colab: using the configured local paths.')
if WINDOW_SECONDS <= 0 or CHUNK_ROWS <= 0 or not 0 < MIN_COVERAGE <= 1:
    raise ValueError('Window/chunk sizes must be positive and MIN_COVERAGE must be in (0,1].')
if MODE not in {'real','demo'}:
    raise ValueError('MODE must be real or demo.')

## 3 · Analysis engine
The following implementation cells are collapsed in Colab. You can expand them to inspect every transformation. They define functions only; the numbered execution cells below run the analysis. No other code files or repository cloning are needed.

In [ ]:
# @title Engine 1. Imports, configuration and audit helpers
from pathlib import Path
from dataclasses import dataclass, asdict
import base64, hashlib, html, importlib.metadata, json, math, os, re, shutil, sys, time, zipfile
from datetime import datetime, timezone
import warnings
import numpy as np
import pandas as pd
import h5py
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
from scipy.stats import entropy
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display, HTML
except ImportError:  # The Colab runtime includes IPython; allow local CLI validation too.
    def display(x): print(x)
    def HTML(x): return x

VERSION = '1.0.0'
META_COLS = ['organoid_id','experiment_id','role','substrate','creation_utc','age_basis',
 'recording_start_utc','recording_end_utc','coverage_verified','channel_ids',
 'endpoint_utc','endpoint_kind','endpoint_verified','endpoint_evidence']
SOURCE_COLS = ['path','experiment_id','organoid_id','format','hdf_key','time_col',
 'channel_col','organoid_col','amplitude_col','time_kind','origin_utc',
 'sampling_rate_hz','waveform_key']
NEURAL_GROUPS = {
 'rates':['population_rate_hz','mean_channel_rate_hz','std_channel_rate_hz','active_fraction','rate_gini'],
 'timing':['isi_median_s','isi_cv','isi_cv2','fraction_isi_under_10ms'],
 'bursts':['channel_burst_rate_min','fraction_spikes_in_channel_bursts','network_burst_rate_min','network_burst_occupancy'],
 'population':['population_fano_1s','count_entropy_1s','pairwise_corr_1s','synchrony_100ms'],
 'amplitude':['amplitude_abs_median','waveform_p2p_median','waveform_width_us_median'],
}
ENV_FEATURES = ['temperature_c_mean','temperature_c_std','co2_pct_mean','humidity_pct_mean','flow_rate_mean']
NEURAL_FEATURES = [x for v in NEURAL_GROUPS.values() for x in v]
PALETTE = ['#064c37','#d28a42','#5b7c69','#758a99','#b26c6c']
plt.rcParams.update({'figure.dpi':110,'savefig.dpi':180,'axes.spines.top':False,
 'axes.spines.right':False,'axes.prop_cycle':matplotlib.cycler(color=PALETTE),
 'font.size':10,'figure.facecolor':'white','axes.titlesize':12})

@dataclass
class Config:
    data_root: str
    output_root: str
    mode: str = 'real'
    window_seconds: int = 3600
    chunk_rows: int = 200_000
    max_window_events: int = 2_000_000
    min_coverage: float = 0.95
    source_timezone: str = 'UTC'
    seed: int = 42
    run_ablations: bool = True
    run_shap: bool = False
    bootstrap_repeats: int = 300
    max_files: int = 50_000
    min_training_organoids: int = 3
    endpoint_definition: str = 'confirmed_death'

def log(msg):
    print(f'[{datetime.now(timezone.utc).strftime("%H:%M:%S UTC")}] {msg}', flush=True)

def initialize_output(cfg):
    """Remove only known derived outputs, preserving caches and edited setup templates."""
    root=Path(cfg.data_root).resolve();out=Path(cfg.output_root).resolve()
    if root==out:raise ValueError('DATA_ROOT and OUTPUT_ROOT must be different folders.')
    out.mkdir(parents=True,exist_ok=True)
    generated={
      'file_inventory.csv','sources_suggested.csv','metadata_audit.csv','ingestion_audit.json',
      'hourly_features.parquet','window_quality_audit.csv','analysis_status.json',
      'held_out_predictions.csv','model_comparison.csv','per_organoid_errors.csv',
      'skipped_analyses.json','feature_ablations.csv','held_out_feature_importance.csv',
      'environment_mixed_effects.csv','survival_landmark_table.csv','kaplan_meier.csv',
      'cox_exploratory.csv','secondary_analysis_status.json','external_descriptive_features.csv',
      'external_human_age_predictions.csv','rat_separate_task_predictions.csv',
      'run_configuration.json','environment_versions.json','BDAPP_REPORT.html',
      'INTERPRETATION.txt','BDAPP_PRIVATE_RESULTS.zip','public_recording_features.csv',
      'public_adapter_issues.json','endpoint_activity_associations.csv'}
    for p in out.iterdir():
        if p.is_file() and (p.name in generated or (p.name.startswith('figure_') and p.suffix=='.png')):p.unlink()
    log('New output run initialized; reusable cache and setup templates retained.')

def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str),encoding='utf8');tmp.replace(path)

def fingerprint(path):
    p=Path(path);s=p.stat();h=hashlib.sha256()
    with p.open('rb') as f:
        h.update(f.read(65536))
        if s.st_size>65536:f.seek(max(0,s.st_size-65536));h.update(f.read(65536))
    return {'path':str(p.resolve()),'bytes':s.st_size,'mtime_ns':s.st_mtime_ns,'edge_sha256':h.hexdigest()}

def bool_value(x):return str(x).strip().lower() in {'true','1','yes','y'}

def utc(x,tz='UTC'):
    if pd.isna(x) or str(x).strip()=='':return pd.NaT
    t=pd.Timestamp(x)
    return t.tz_localize(tz).tz_convert('UTC') if t.tzinfo is None else t.tz_convert('UTC')

def text_col(series):
    return series.map(lambda x:x.decode('utf8') if isinstance(x,(bytes,np.bytes_)) else str(x))

def guess_experiment(path):
    x=re.search(r'fs[_ -]?(\d+)',str(path),flags=re.I)
    return 'FS'+x.group(1) if x else ''

def infer_time_kind(values):
    s=pd.Series(values).dropna()
    if not len(s):return ''
    numeric=pd.to_numeric(s,errors='coerce')
    if numeric.notna().mean()<.5:
        return 'iso' if pd.to_datetime(text_col(s),errors='coerce',utc=True).notna().mean()>.95 else ''
    m=float(numeric.median())
    for k,scale in [('unix_s',1),('unix_ms',1e3),('unix_us',1e6),('unix_ns',1e9)]:
        if 946684800<=m/scale<=4102444800:return k
    return ''  # Relative units/origin cannot be inferred safely.

def to_epoch_seconds(values,kind,origin='',sampling_rate_hz='',tz='UTC'):
    s=pd.Series(values)
    if kind=='iso':
        def conv(v):
            try:return utc(v,tz).timestamp()
            except Exception:return np.nan
        return s.map(conv).to_numpy(float)
    scales={'unix_s':1,'unix_ms':1e3,'unix_us':1e6,'unix_ns':1e9,
            'relative_s':1,'relative_ms':1e3,'relative_us':1e6,'relative_ns':1e9}
    a=pd.to_numeric(s,errors='coerce').to_numpy(float)
    if kind=='samples':
        rate=float(sampling_rate_hz)
        if rate<=0:raise ValueError('A positive sampling_rate_hz is required for sample indices.')
        a=a/rate
    elif kind in scales:a=a/scales[kind]
    else:raise ValueError(f'Unknown time_kind {kind!r}. Set ISO/Unix units or relative units plus origin_utc in sources.csv.')
    if kind.startswith('relative_') or kind=='samples':
        if pd.isna(utc(origin,tz)):raise ValueError('Relative timestamps require a verified origin_utc, not a guessed creation date.')
        a=a+utc(origin,tz).timestamp()
    finite=a[np.isfinite(a)]
    if len(finite) and (finite.min()<946684800 or finite.max()>4102444800):
        raise ValueError('Converted timestamps fall outside 2000-2100. Verify units and origin.')
    return a

In [ ]:
# @title Engine 2. Inspect files without loading full recordings
ALIASES={
 'time':['timestamp_utc','timestamp','time','timestamps','spike_time','spike_times'],
 'channel':['channel','channel_id','electrode','electrode_id'],
 'organoid':['organoid_id','culture_id','organoid','culture'],
 'amplitude':['max amplitude','amplitude','amplitude_uv','peak_amplitude']}

def pick(columns,kind):
    matches=[c for c in columns if str(c).lower().strip() in ALIASES[kind]]
    return str(matches[0]) if len(matches)==1 else ''

def hdf_tables(path):
    found=[]
    with h5py.File(path,'r') as f:
        def visit(name,obj):
            if isinstance(obj,h5py.Dataset) and obj.ndim==1 and obj.dtype.names:
                cols=list(obj.dtype.names)
                if pick(cols,'time') and pick(cols,'channel'):
                    a=obj[:min(20,len(obj))]
                    found.append((name,cols,a[pick(cols,'time')]))
            if isinstance(obj,h5py.Group):
                cols=[k for k,v in obj.items() if isinstance(v,h5py.Dataset) and v.ndim==1]
                if pick(cols,'time') and pick(cols,'channel'):
                    tk,ck=pick(cols,'time'),pick(cols,'channel')
                    if len(obj[tk])==len(obj[ck]):found.append((name,cols,obj[tk][:20]))
        visit('',f);f.visititems(visit)
    return found

def inspect_files(cfg):
    root=Path(cfg.data_root);out=Path(cfg.output_root);out.mkdir(parents=True,exist_ok=True)
    if not root.exists():raise FileNotFoundError(f'Data folder does not exist: {root}. Set DATA_ROOT to your extracted folder.')
    if cfg.mode=='real' and (root/'SYNTHETIC_DATA_ONLY.txt').exists():raise ValueError('Synthetic fixture detected in real mode. Choose your real data folder or explicitly set MODE=demo.')
    reserved={'organoids.csv','sources.csv','channel_map.csv','coverage.csv','events.csv','environment.csv'}
    rows=[];profiles=[];issues=[]
    for p in root.rglob('*'):
        if len(rows)>=cfg.max_files:raise RuntimeError('File scan limit reached. Point DATA_ROOT at a narrower dataset folder.')
        if not p.is_file() or out.resolve() in p.resolve().parents or p.name in reserved:continue
        if p.suffix.lower() not in {'.csv','.tsv','.parquet','.h5','.hdf5','.hdf','.nwb','.mat','.zip'}:continue
        rec={'path':str(p.relative_to(root)),'size_gb':p.stat().st_size/1e9,'status':'needs_mapping','details':''}
        candidates=[]
        try:
            if p.suffix.lower() in {'.csv','.tsv'}:
                a=pd.read_csv(p,nrows=20,sep='\t' if p.suffix.lower()=='.tsv' else ',')
                candidates=[('',list(a.columns),a[pick(a.columns,'time')]) ] if pick(a.columns,'time') and pick(a.columns,'channel') else []
                fmt='tsv' if p.suffix.lower()=='.tsv' else 'csv'
            elif p.suffix.lower()=='.parquet':
                pf=pq.ParquetFile(p);a=next(pf.iter_batches(batch_size=20)).to_pandas()
                candidates=[('',list(a.columns),a[pick(a.columns,'time')])] if pick(a.columns,'time') and pick(a.columns,'channel') else []
                fmt='parquet'
            elif p.suffix.lower() in {'.h5','.hdf5','.hdf'}:
                candidates=hdf_tables(p);fmt='hdf5'
                if not candidates:
                    with h5py.File(p,'r') as f:
                        info=[]
                        f.visititems(lambda k,v:info.append(f'{k}: {v.shape} {v.dtype}') if isinstance(v,h5py.Dataset) and len(info)<30 else None)
                    rec['details']=' | '.join(info)
            else:
                fmt='unsupported';rec['details']='MAT/NWB are handled separately as external descriptive recordings. Extract ZIP files first. They are never silently pooled with FS training data.'
            if len(candidates)==1:
                key,cols,times=candidates[0]
                profile={c:'' for c in SOURCE_COLS}
                profile.update(path=rec['path'],experiment_id=guess_experiment(p),format=fmt,hdf_key=key,
                 time_col=pick(cols,'time'),channel_col=pick(cols,'channel'),organoid_col=pick(cols,'organoid'),
                 amplitude_col=pick(cols,'amplitude'),time_kind=infer_time_kind(times))
                profiles.append(profile);rec['status']='recognized';rec['details']=str(cols)
                if not profile['time_kind']:rec['status']='set_time_units'
            elif len(candidates)>1:rec['details']='Multiple event tables: '+str([v[0] for v in candidates])
        except Exception as e:rec['details']=type(e).__name__+': '+str(e)[:250]
        rows.append(rec)
    inv=pd.DataFrame(rows,columns=['path','size_gb','status','details'])
    inv.to_csv(out/'file_inventory.csv',index=False)
    proposed=pd.DataFrame(profiles,columns=SOURCE_COLS)
    proposed.to_csv(out/'sources_suggested.csv',index=False)
    return inv,proposed

def read_csv_optional(path,columns):
    return pd.read_csv(path).fillna('') if Path(path).exists() else pd.DataFrame(columns=columns)

def prepare_project(cfg,proposed):
    root=Path(cfg.data_root);out=Path(cfg.output_root)
    meta_path=root/'organoids.csv';source_path=root/'sources.csv'
    if not meta_path.exists():
        template=pd.DataFrame(columns=META_COLS)
        if not (out/'organoids_TEMPLATE.csv').exists():template.to_csv(out/'organoids_TEMPLATE.csv',index=False)
        if not source_path.exists():proposed.to_csv(out/'sources_TEMPLATE.csv',index=False)
        raise ValueError('Setup needed: fill organoids_TEMPLATE.csv with the real culture IDs, dates, channel IDs and verified coverage; save as DATA_ROOT/organoids.csv. Source suggestions are in sources_TEMPLATE.csv. No dates or endpoints are invented.')
    meta=pd.read_csv(meta_path).fillna('')
    for c in META_COLS:
        if c not in meta:meta[c]=''
    if meta.empty or meta.organoid_id.eq('').any() or meta.organoid_id.duplicated().any():raise ValueError('organoids.csv needs unique, nonempty, GLOBAL organoid_id values such as FS369-org1.')
    if meta.experiment_id.eq('').any():raise ValueError('Every organoid needs an experiment_id.')
    meta['role']=meta.role.replace('', 'primary')
    if not set(meta.role)<= {'primary','external'}:raise ValueError('role must be primary or external.')
    for c in ['creation_utc','recording_start_utc','recording_end_utc','endpoint_utc']:meta[c]=meta[c].map(lambda x:utc(x,cfg.source_timezone))
    for c in ['coverage_verified','endpoint_verified']:meta[c]=meta[c].map(bool_value)
    for _,r in meta.iterrows():
        if pd.isna(r.recording_start_utc) or pd.isna(r.recording_end_utc) or r.recording_start_utc>=r.recording_end_utc:raise ValueError(f'{r.organoid_id}: valid recording_start_utc / recording_end_utc required.')
        if pd.notna(r.creation_utc) and r.creation_utc>r.recording_start_utc:raise ValueError(f'{r.organoid_id}: creation date is later than recording start.')
        if pd.notna(r.creation_utc) and r.age_basis not in {'biological_creation','days_since_first_recording'}:raise ValueError('age_basis must explicitly be biological_creation or days_since_first_recording when creation_utc is supplied.')
        if r.role=='primary' and r.substrate!='human_organoid':raise ValueError('Primary FS age models require substrate=human_organoid. Put rat/external recordings in role=external.')
        if not str(r.channel_ids).strip():raise ValueError(f'{r.organoid_id}: channel_ids required (semicolon-separated, e.g. 0;1;2;3;4;5;6;7). Do not assume zero- or one-based indexing.')
        if r.endpoint_verified and (r.endpoint_kind not in {'confirmed_death','administrative_censoring'} or pd.isna(r.endpoint_utc) or not r.endpoint_evidence):raise ValueError('A verified endpoint needs timestamp, confirmed_death or administrative_censoring, and endpoint_evidence.')
        if pd.notna(r.endpoint_utc) and r.endpoint_utc<r.recording_start_utc:raise ValueError('Endpoint precedes first recording.')
    sources=read_csv_optional(source_path,SOURCE_COLS) if source_path.exists() else proposed.copy()
    for c in SOURCE_COLS:
        if c not in sources:sources[c]=''
    if sources.empty:raise ValueError('No usable spike sources. Review file_inventory.csv and fill sources.csv with an explicit event-table mapping.')
    for _,r in sources.iterrows():
        p=root/str(r.path)
        if not p.is_file():raise FileNotFoundError(str(p))
        if not r.experiment_id:raise ValueError(f'{r.path}: experiment_id not identified. Set it in sources.csv.')
        if not r.time_kind:raise ValueError(f'{r.path}: set time_kind and, for relative times, origin_utc in sources.csv.')
    if sources.path.duplicated().any():raise ValueError('Each source path must appear once; combine multiple HDF event groups in separate exports before use.')
    meta['channels']=meta.channel_ids.map(lambda s:[str(v).strip() for v in str(s).split(';') if str(v).strip()])
    for _,r in meta.iterrows():
        if len(set(r.channels))!=len(r.channels):raise ValueError(f'Duplicate channel_ids for {r.organoid_id}')
    cmap=read_csv_optional(root/'channel_map.csv',['experiment_id','channel','organoid_id'])
    if len(cmap):
        if cmap.duplicated(['experiment_id','channel']).any():raise ValueError('channel_map.csv contains ambiguous duplicate experiment/channel mappings.')
        if not set(cmap.organoid_id)<=set(meta.organoid_id):raise ValueError('Unknown organoid in channel_map.csv.')
    coverage=read_csv_optional(root/'coverage.csv',['organoid_id','start_utc','end_utc'])
    intervals={}
    for _,r in meta.iterrows():
        cr=coverage[coverage.organoid_id==r.organoid_id]
        vals=[(utc(a,cfg.source_timezone).timestamp(),utc(b,cfg.source_timezone).timestamp()) for a,b in zip(cr.start_utc,cr.end_utc)] if len(cr) else ([(r.recording_start_utc.timestamp(),r.recording_end_utc.timestamp())] if r.coverage_verified else [])
        merged=[]
        for a,b in sorted(vals):
            if b<=a:raise ValueError('Coverage interval ends before it starts.')
            if a<r.recording_start_utc.timestamp() or b>r.recording_end_utc.timestamp():raise ValueError('Coverage interval lies outside declared recording bounds.')
            if merged and a<=merged[-1][1]:merged[-1]=(merged[-1][0],max(b,merged[-1][1]))
            else:merged.append((a,b))
        intervals[r.organoid_id]=merged
    events=read_csv_optional(root/'events.csv',['experiment_id','organoid_id','timestamp_utc','event_type','exclude_before_min','exclude_after_min'])
    meta.to_csv(out/'metadata_audit.csv',index=False)
    return {'meta':meta,'sources':sources,'channel_map':cmap,'coverage':intervals,'events':events,'cfg':cfg}

In [ ]:
# @title Engine 3. Chunked ingestion and restart-safe source cache
def source_batches(path,spec,chunk_rows):
    fmt=spec['format']
    if fmt in {'csv','tsv'}:
        cols=[str(spec[k]) for k in ['time_col','channel_col','organoid_col','amplitude_col'] if str(spec.get(k,''))]
        for a in pd.read_csv(path,sep='\t' if fmt=='tsv' else ',',chunksize=chunk_rows,usecols=cols):yield a
    elif fmt=='parquet':
        for b in pq.ParquetFile(path).iter_batches(batch_size=chunk_rows):yield b.to_pandas()
    elif fmt=='hdf5':
        with h5py.File(path,'r') as f:
            obj=f[str(spec.get('hdf_key','')) or '/']
            if isinstance(obj,h5py.Dataset):
                if not obj.dtype.names or obj.ndim!=1:raise ValueError('HDF dataset must be a 1D compound event table; raw voltage matrices are not spike events.')
                cols=[str(spec[k]) for k in ['time_col','channel_col','organoid_col','amplitude_col'] if str(spec.get(k,''))]
                for i in range(0,len(obj),chunk_rows):
                    raw=obj[i:i+chunk_rows];a=pd.DataFrame({c:raw[c] for c in cols})
                    wk=str(spec.get('waveform_key',''))
                    if wk:
                        w=raw[wk] if wk in obj.dtype.names else f[wk][i:i+chunk_rows]
                        rate=float(spec.get('sampling_rate_hz',0))
                        if w.ndim!=2 or len(w)!=len(a) or rate<=0:raise ValueError('Waveforms must align with spike rows and need sampling_rate_hz.')
                        a['_p2p']=np.ptp(w,axis=1);a['_width_us']=np.abs(np.argmax(w,axis=1)-np.argmin(w,axis=1))*1e6/rate
                    yield a
            else:
                cols=[str(spec[k]) for k in ['time_col','channel_col','organoid_col','amplitude_col'] if str(spec.get(k,''))]
                n=len(obj[str(spec['time_col'])])
                if any(len(obj[c])!=n for c in cols):raise ValueError('HDF column lengths differ.')
                for i in range(0,n,chunk_rows):
                    a=pd.DataFrame({c:obj[c][i:i+chunk_rows] for c in cols})
                    wk=str(spec.get('waveform_key',''))
                    if wk:
                        w=f[wk][i:i+chunk_rows]
                        rate=float(spec.get('sampling_rate_hz',0))
                        if w.ndim!=2 or len(w)!=len(a) or rate<=0:raise ValueError('Waveforms must align one-to-one with spikes and need sampling_rate_hz.')
                        a['_p2p']=np.ptp(w,axis=1)
                        a['_width_us']=np.abs(np.argmax(w,axis=1)-np.argmin(w,axis=1))*1e6/rate
                    yield a
    else:raise ValueError(f'Unsupported format: {fmt}. Supported event exports: csv, tsv, parquet, columnar/compound HDF5.')

def normalize_events(a,spec,project):
    cfg=project['cfg'];meta=project['meta'];experiment=str(spec['experiment_id'])
    required=[spec['time_col'],spec['channel_col']]
    if any(c not in a for c in required):raise ValueError(f'Missing columns {required}. Available: {list(a.columns)}')
    b=pd.DataFrame({'timestamp_s':to_epoch_seconds(a[spec['time_col']],str(spec['time_kind']),spec.get('origin_utc',''),spec.get('sampling_rate_hz',''),cfg.source_timezone),
      'channel':text_col(a[spec['channel_col']]).map(lambda s:re.sub(r'\.0$','',s))})
    orgcol=str(spec.get('organoid_col',''))
    if orgcol:
        if orgcol not in a:raise ValueError(f'Missing organoid column {orgcol}')
        b['organoid_id']=text_col(a[orgcol])
    elif str(spec.get('organoid_id','')):b['organoid_id']=str(spec['organoid_id'])
    else:
        cm=project['channel_map'];cm=cm[cm.experiment_id==experiment]
        if cm.empty:
            mm=meta[meta.experiment_id==experiment]
            if len(mm)!=1:raise ValueError(f'{experiment}: spike source has multiple organoids but no organoid column or channel_map.csv. Do not guess 0/1-based channel slots.')
            b['organoid_id']=mm.iloc[0].organoid_id
        else:
            mapping={re.sub(r'\.0$','',str(c)):o for c,o in zip(cm.channel,cm.organoid_id)}
            b['organoid_id']=b.channel.map(mapping)
    if b.organoid_id.isna().any():raise ValueError('Unmapped channels in an event source. Add every observed channel to channel_map.csv.')
    allowed=meta.set_index('organoid_id')
    if not set(b.organoid_id)<=set(allowed.index):raise ValueError('Unknown organoid_id in spike events; IDs must match organoids.csv.')
    for oid,g in b.groupby('organoid_id'):
        r=allowed.loc[oid]
        if r.experiment_id!=experiment:raise ValueError('Source experiment conflicts with organoid metadata.')
        if not set(g.channel)<=set(r.channels):raise ValueError(f'{oid}: observed channel IDs do not match declared channel_ids.')
    for col,key in [('amplitude','amplitude_col')]:
        b[col]=pd.to_numeric(a[str(spec[key])],errors='coerce').to_numpy(float) if str(spec.get(key,'')) else np.nan
    b['waveform_p2p']=pd.to_numeric(a.get('_p2p',pd.Series(np.nan,index=a.index)),errors='coerce').to_numpy(float)
    b['waveform_width_us']=pd.to_numeric(a.get('_width_us',pd.Series(np.nan,index=a.index)),errors='coerce').to_numpy(float)
    if not np.isfinite(b.timestamp_s).all():raise ValueError('Some timestamps cannot be parsed. Fix source mapping instead of silently dropping records.')
    b['experiment_id']=experiment
    return b[['experiment_id','organoid_id','channel','timestamp_s','amplitude','waveform_p2p','waveform_width_us']]

def ingest(project):
    cfg=project['cfg'];root=Path(cfg.data_root);cache=Path(cfg.output_root)/'cache/events';cache.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(cache).free<1_000_000_000:raise RuntimeError('Less than 1 GB free in output storage. Free space before ingestion.')
    paths=[];audit=[]
    mapping_hash=digest({'metadata':project['meta'].to_json(date_format='iso'),'map':project['channel_map'].to_json(),'version':VERSION,'timezone':cfg.source_timezone})
    for idx,(_,r) in enumerate(project['sources'].iterrows(),1):
        spec=r.to_dict();p=root/str(r.path);fp=fingerprint(p)
        key=digest({'file':fp,'spec':spec,'mapping':mapping_hash});dest=cache/f'{key}.parquet';done=cache/f'{key}.json'
        if dest.exists() and done.exists():
            info=json.loads(done.read_text());log(f'Reusing {idx}/{len(project["sources"])}: {p.name}')
        else:
            log(f'Reading {idx}/{len(project["sources"])}: {p.name} in chunks')
            temp=dest.with_suffix('.partial');writer=None;total=0
            try:
                for chunk_number,a in enumerate(source_batches(p,spec,cfg.chunk_rows),1):
                    b=normalize_events(a,spec,project)
                    if b.empty:continue
                    b=b.sort_values(['organoid_id','timestamp_s'])
                    tab=pa.Table.from_pandas(b,preserve_index=False)
                    if writer is None:writer=pq.ParquetWriter(temp,tab.schema,compression='zstd')
                    writer.write_table(tab);total+=len(b)
                    if chunk_number%10==0:log(f'  {p.name}: {total:,} spike events normalized')
                if writer:writer.close();writer=None;temp.replace(dest)
                else:raise ValueError('Source contained no spike events.')
                info={'key':key,'input':fp,'normalized_rows':total,'deduplication':'performed within each feature window across sources','profile':spec}
                atomic_json(done,info)
            except Exception:
                if writer:writer.close()
                temp.unlink(missing_ok=True);raise
        paths.append(str(dest));audit.append(info)
    atomic_json(Path(cfg.output_root)/'ingestion_audit.json',audit)
    con=duckdb.connect();con.execute('SET threads=2')
    spill=Path('/content/bdapp_duckdb_spill') if Path('/content').exists() else Path(cfg.output_root)/'cache/spill'
    spill.mkdir(parents=True,exist_ok=True)
    con.execute('SET temp_directory=?',[str(spill)])
    con.from_parquet(paths).create_view('spikes')
    return con,paths,digest({'paths':paths,'coverage':project['coverage'],'meta':mapping_hash,'window_seconds':cfg.window_seconds,'events':project['events'].to_json(),'version':VERSION})

In [ ]:
# @title Engine 4. Hourly exposure, artifacts and neural feature extraction
def overlap_seconds(a,b,intervals):return sum(max(0,min(b,y)-max(a,x)) for x,y in intervals)

def event_flags(row,a,b,events,cfg):
    flags=[]
    for _,e in events.iterrows():
        if str(e.get('experiment_id','')) not in {'',str(row.experiment_id)}:continue
        if str(e.get('organoid_id','')) not in {'',str(row.organoid_id)}:continue
        t=utc(e.timestamp_utc,cfg.source_timezone)
        if pd.isna(t):raise ValueError('events.csv contains a missing/invalid event timestamp.')
        before=float(e.get('exclude_before_min','') or 0)*60
        after=float(e.get('exclude_after_min','') or 0)*60
        if before<0 or after<0:raise ValueError('Event exclusion durations must be nonnegative.')
        # Half-open windows; a zero-duration event is assigned to its containing window.
        if (a<=t.timestamp()<b) or (a<t.timestamp()+after and b>t.timestamp()-before):flags.append(str(e.event_type))
    return sorted(set(flags))

def neural_features(a,channels,start,stop,exposure):
    out={k:np.nan for k in NEURAL_FEATURES}
    if exposure<=0:return out
    duration=stop-start;counts=np.array([(a.channel==c).sum() for c in channels],float)
    rates=counts/exposure;out.update(population_rate_hz=len(a)/exposure,mean_channel_rate_hz=rates.mean(),std_channel_rate_hz=rates.std(),active_fraction=float((counts>0).mean()))
    total=counts.sum()
    out['rate_gini']=float(np.abs(counts[:,None]-counts[None,:]).sum()/(2*len(counts)*total)) if total else 0.
    isis=[];cv=[];cv2=[];burst_n=0;burst_spikes=0
    bin_count=int(math.ceil(duration));coarse=np.zeros((len(channels),bin_count),dtype=np.int32)
    fine_n=int(math.ceil(duration/.1));fine=np.zeros((len(channels),fine_n),dtype=bool)
    for j,c in enumerate(channels):
        t=np.sort(a.loc[a.channel==c,'timestamp_s'].to_numpy(float))
        isi=np.diff(t)
        # Cross-gap ISIs must not masquerade as long interspike intervals.
        # Timing and binned metrics are only reported for fully observed windows.
        if len(isi):
            isis.extend(isi.tolist())
            if isi.mean()>0:cv.append(float(isi.std()/isi.mean()))
            if len(isi)>1:
                den=isi[:-1]+isi[1:];ok=den>0
                if ok.any():cv2.append(float(np.mean(2*np.abs(np.diff(isi))[ok]/den[ok])))
            links=isi<=.1
            edges=np.diff(np.r_[False,links,False].astype(int))
            for x,y in zip(np.where(edges==1)[0],np.where(edges==-1)[0]):
                n=y-x+1
                if n>=5:burst_n+=1;burst_spikes+=n
        if len(t):
            ix=np.clip(((t-start)/1.).astype(int),0,bin_count-1);np.add.at(coarse[j],ix,1)
            fine[j,np.clip(((t-start)/.1).astype(int),0,fine_n-1)]=True
    fully=abs(exposure-duration)<1e-4
    if fully:
        if isis:
            isis=np.asarray(isis);out['isi_median_s']=float(np.median(isis));out['fraction_isi_under_10ms']=float((isis<.01).mean())
        out['isi_cv']=float(np.mean(cv)) if cv else np.nan
        out['isi_cv2']=float(np.mean(cv2)) if cv2 else np.nan
        out['channel_burst_rate_min']=burst_n/(duration/60)/len(channels)
        out['fraction_spikes_in_channel_bursts']=burst_spikes/len(a) if len(a) else 0.
        net=fine.sum(axis=0)>=max(2,int(math.ceil(.75*len(channels))))
        out['network_burst_rate_min']=float(np.count_nonzero(np.diff(np.r_[False,net].astype(int))==1)/(duration/60))
        out['network_burst_occupancy']=float(net.mean())
        pop=coarse.sum(axis=0)
        out['population_fano_1s']=float(pop.var()/pop.mean()) if pop.mean()>0 else 0.
        freq=np.unique(pop,return_counts=True)[1]
        out['count_entropy_1s']=float(entropy(freq,base=2))
        active=np.std(coarse,axis=1)>0
        if active.sum()>1:
            cc=np.corrcoef(coarse[active]);out['pairwise_corr_1s']=float(cc[np.triu_indices(len(cc),1)].mean())
        occupied=fine.sum(axis=0)
        out['synchrony_100ms']=float((occupied>=2).sum()/(occupied>0).sum()) if (occupied>0).any() else 0.
    for src,target in [('amplitude','amplitude_abs_median'),('waveform_p2p','waveform_p2p_median'),('waveform_width_us','waveform_width_us_median')]:
        v=a[src].dropna().to_numpy(float)
        if len(v):out[target]=float(np.median(np.abs(v)))
    return out

def environmental_features(project):
    cfg=project['cfg'];p=Path(cfg.data_root)/'environment.csv'
    if not p.exists():return pd.DataFrame(),False
    acc={};variables=['temperature_c','co2_pct','humidity_pct','flow_rate']
    for chunk in pd.read_csv(p,chunksize=cfg.chunk_rows):
        if not {'timestamp_utc','experiment_id'}<=set(chunk):raise ValueError('environment.csv needs timestamp_utc and experiment_id.')
        ts=chunk.timestamp_utc.map(lambda x:utc(x,cfg.source_timezone).timestamp())
        chunk['window_s']=(ts//cfg.window_seconds*cfg.window_seconds).astype('int64')
        for (exp,w),g in chunk.groupby(['experiment_id','window_s']):
            d=acc.setdefault((str(exp),int(w)),{})
            for col in variables:
                if col not in g:continue
                v=pd.to_numeric(g[col],errors='coerce').dropna().to_numpy(float)
                if len(v):
                    z=d.setdefault(col,[0,0.,0.]);z[0]+=len(v);z[1]+=v.sum();z[2]+=(v*v).sum()
    rows=[]
    for (exp,w),data in acc.items():
        r={'experiment_id':exp,'window_start_s':w}
        for col,(n,s,ss) in data.items():r[col+'_mean']=s/n;r[col+'_std']=math.sqrt(max(0,ss/n-(s/n)**2))
        rows.append(r)
    return pd.DataFrame(rows),True

def extract_features(project,con,ingest_key):
    cfg=project['cfg'];out=Path(cfg.output_root);cache=out/'cache/features';cache.mkdir(parents=True,exist_ok=True)
    key=digest({'ingest':ingest_key,'config':asdict(cfg),'definitions':'fixed-ISI-100ms-5spikes;network-75pct-fixed-100ms-v1'})
    parts=[];audits=[]
    for _,r in project['meta'].iterrows():
        oid=r.organoid_id;part=cache/f'{digest([key,oid])}.parquet';done=part.with_suffix('.json')
        if part.exists() and done.exists():parts.append(pd.read_parquet(part));audits.extend(json.loads(done.read_text()));log(f'Reusing features: {oid}');continue
        start=r.recording_start_utc.timestamp();stop=r.recording_end_utc.timestamp()
        outside=con.execute('SELECT count(*) FROM spikes WHERE organoid_id=? AND (timestamp_s<? OR timestamp_s>=?)',[oid,start,stop]).fetchone()[0]
        if outside:raise ValueError(f'{oid}: {outside} events lie outside recording bounds. Verify clock, units, mapping, and exclusive recording_end_utc.')
        n_windows=int(math.ceil(stop/cfg.window_seconds)-math.floor(start/cfg.window_seconds))
        if n_windows>20000:raise ValueError('Implausibly many windows; verify units and recording bounds.')
        rows=[];qa=[]
        for a in np.arange(math.floor(start/cfg.window_seconds)*cfg.window_seconds,stop,cfg.window_seconds):
            b=a+cfg.window_seconds
            n_raw=con.execute('SELECT count(*) FROM spikes WHERE organoid_id=? AND timestamp_s>=? AND timestamp_s<?',[oid,a,b]).fetchone()[0]
            if n_raw>cfg.max_window_events:raise MemoryError(f'{oid}: {n_raw:,} events in one window exceeds MAX_WINDOW_EVENTS. Reduce WINDOW_SECONDS or raise the limit with enough RAM; no events are subsampled.')
            frame=con.execute('SELECT channel,timestamp_s,avg(amplitude) AS amplitude,avg(waveform_p2p) AS waveform_p2p,avg(waveform_width_us) AS waveform_width_us FROM spikes WHERE organoid_id=? AND timestamp_s>=? AND timestamp_s<? GROUP BY channel,timestamp_s ORDER BY timestamp_s',[oid,a,b]).fetchdf()
            intervals=project['coverage'][oid];exposure=overlap_seconds(a,b,intervals)
            covered=np.zeros(len(frame),dtype=bool)
            for x,y in intervals:covered|=(frame.timestamp_s.to_numpy()>=x)&(frame.timestamp_s.to_numpy()<y)
            uncovered=int((~covered).sum()) if intervals else 0
            feats=neural_features(frame.loc[covered].copy(),r.channels,a,b,exposure)
            flags=event_flags(r,a,b,project['events'],cfg)
            coverage=exposure/cfg.window_seconds
            ok=coverage>=cfg.min_coverage and not flags and uncovered==0
            mid=a+cfg.window_seconds/2
            # Predict at the END of the preceding window; no activity after the forecast time enters features.
            age=(b-r.creation_utc.timestamp())/86400 if pd.notna(r.creation_utc) else np.nan
            death=bool(r.endpoint_verified and r.endpoint_kind=='confirmed_death')
            if death and pd.notna(r.endpoint_utc) and b>r.endpoint_utc.timestamp():ok=False
            remaining=(r.endpoint_utc.timestamp()-b)/86400 if death and pd.notna(r.endpoint_utc) and b<=r.endpoint_utc.timestamp() else np.nan
            row={'organoid_id':oid,'experiment_id':r.experiment_id,'role':r.role,'substrate':r.substrate,'age_basis':r.age_basis,
                 'window_start_s':int(a),'window_start_utc':pd.to_datetime(a,unit='s',utc=True),'window_mid_s':mid,'prediction_time_s':b,'duration_s':cfg.window_seconds,
                 'coverage_seconds':exposure,'coverage_fraction':coverage,'expected_channels':len(r.channels),'n_spikes':len(frame),
                 'duplicates_removed':n_raw-len(frame),'events_outside_coverage':uncovered,'exclusion_events':';'.join(flags),'qc_pass':ok,
                 'age_days':age,'remaining_life_days':remaining,'elapsed_recording_days':(b-start)/86400,**feats}
            rows.append(row);qa.append({k:row[k] for k in ['organoid_id','window_start_utc','qc_pass','coverage_fraction','exclusion_events','duplicates_removed','events_outside_coverage']})
        f=pd.DataFrame(rows);f.to_parquet(part,index=False);atomic_json(done,qa);parts.append(f);audits.extend(qa)
        log(f'Features ready: {oid} ({len(rows):,} windows)')
    f=pd.concat(parts,ignore_index=True)
    env,env_present=environmental_features(project)
    if len(env):f=f.merge(env,on=['experiment_id','window_start_s'],how='left',validate='many_to_one')
    for c in ENV_FEATURES:
        if c not in f:f[c]=np.nan
    f=f.replace([np.inf,-np.inf],np.nan)
    f.to_parquet(out/'hourly_features.parquet',index=False);pd.DataFrame(audits).to_csv(out/'window_quality_audit.csv',index=False)
    atomic_json(out/'analysis_status.json',{'mode':cfg.mode,'windows':len(f),'qc_pass':int(f.qc_pass.sum()),'organoids':f.organoid_id.nunique(),
       'environment_logs_present':env_present,'event_log_present':bool(len(project['events'])),'unknown_coverage_organoids':[k for k,v in project['coverage'].items() if not v]})
    return f

In [ ]:
# @title Engine 5. Grouped evaluation, batch transfer and uncertainty
def valid_training_frame(features,target,age_basis='biological_creation'):
    mask=features.qc_pass & features.role.eq('primary') & features.substrate.eq('human_organoid')
    if target=='age_days':mask=mask & features.age_basis.eq(age_basis)
    return features.loc[mask & features[target].notna() & features[target].ge(0)].copy().reset_index(drop=True)

def model_specs(seed):
    linear=lambda:make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),StandardScaler(),Ridge(alpha=10.))
    return {
       'Median':(DummyRegressor(strategy='median'),NEURAL_GROUPS['rates'][:1]),
       'Rate-only Ridge':(linear(),['population_rate_hz']),
       'Neural Ridge':(linear(),NEURAL_FEATURES),
       'Neural HistGB':(make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),HistGradientBoostingRegressor(max_iter=120,max_leaf_nodes=7,min_samples_leaf=10,l2_regularization=1.,learning_rate=.06,early_stopping=False,random_state=seed)),NEURAL_FEATURES),
       'Clock diagnostic':(linear(),['elapsed_recording_days']),
    }

def evaluate_split(data,target,train,test,estimator,cols,label,scheme,fold,seed,shuffle=False):
    tr=data.iloc[train];te=data.iloc[test]
    if set(tr.organoid_id)&set(te.organoid_id):raise AssertionError('Leakage: same organoid in training and test.')
    y=tr[target].to_numpy(float).copy()
    if shuffle:
        rng=np.random.default_rng(seed)
        ids=rng.permutation(tr.organoid_id.unique())
        original=y.copy()
        # Shuffle whole target trajectories between cultures, preserving temporal shape.
        donors=np.roll(ids,1)
        for oid,donor in zip(ids,donors):
            ix=np.flatnonzero(tr.organoid_id.to_numpy()==oid)
            jx=np.flatnonzero(tr.organoid_id.to_numpy()==donor)
            ix=ix[np.argsort(tr.iloc[ix].window_start_s.to_numpy())]
            jx=jx[np.argsort(tr.iloc[jx].window_start_s.to_numpy())]
            y[ix]=np.interp(np.linspace(0,1,len(ix)),np.linspace(0,1,len(jx)),original[jx])
    model=clone(estimator);model.fit(tr[cols],y)
    pred=model.predict(te[cols])
    return pd.DataFrame({'target':target,'scheme':scheme,'fold':fold,'model':label,'organoid_id':te.organoid_id.to_numpy(),
       'experiment_id':te.experiment_id.to_numpy(),'window_start_s':te.window_start_s.to_numpy(),'true':te[target].to_numpy(float),'predicted':pred}),model

def metrics_table(predictions,cfg):
    if predictions.empty:return pd.DataFrame(),pd.DataFrame()
    records=[];per_org=[];rng=np.random.default_rng(cfg.seed)
    for (target,scheme,name),g in predictions.groupby(['target','scheme','model']):
        maes=g.assign(error=(g.true-g.predicted).abs()).groupby('organoid_id').error.mean()
        ci=[np.nan,np.nan]
        if len(maes)>=3 and cfg.bootstrap_repeats:
            bs=[rng.choice(maes.to_numpy(),len(maes),replace=True).mean() for _ in range(cfg.bootstrap_repeats)]
            ci=np.quantile(bs,[.025,.975])
        records.append({'target':target,'scheme':scheme,'model':name,'n_organoids':len(maes),'n_windows':len(g),
         'macro_MAE_days':maes.mean(),'macro_MAE_CI_low':ci[0],'macro_MAE_CI_high':ci[1],
         'pooled_MAE_days':mean_absolute_error(g.true,g.predicted),'pooled_RMSE_days':math.sqrt(mean_squared_error(g.true,g.predicted)),
         'pooled_R2':r2_score(g.true,g.predicted) if len(g)>1 and g.true.std()>0 else np.nan})
        per_org.extend({'target':target,'scheme':scheme,'model':name,'organoid_id':k,'MAE_days':v} for k,v in maes.items())
    return pd.DataFrame(records),pd.DataFrame(per_org)

def run_models(features,project):
    cfg=project['cfg'];out=Path(cfg.output_root);preds=[];skips=[];saved={};specs=model_specs(cfg.seed)
    for target in ['age_days','remaining_life_days']:
        data=valid_training_frame(features,target)
        n=data.organoid_id.nunique()
        if n<cfg.min_training_organoids:
            skips.append({'analysis':target,'reason':f'Only {n} independent eligible cultures; need at least {cfg.min_training_organoids}. Age needs creation dates; remaining life needs verified deaths.'});continue
        logo=LeaveOneGroupOut()
        for fold,(tr,te) in enumerate(logo.split(data,groups=data.organoid_id),1):
            for label,(est,cols) in specs.items():
                pred,_=evaluate_split(data,target,tr,te,est,cols,label,'leave_one_organoid_out',fold,cfg.seed)
                preds.append(pred)
            # Negative control is not an inferential permutation test.
            est,cols=specs['Neural Ridge']
            pred,_=evaluate_split(data,target,tr,te,est,cols,'Culture-curve shuffle control','leave_one_organoid_out',fold,cfg.seed+fold,shuffle=True);preds.append(pred)
        train=np.flatnonzero(data.experiment_id.str.upper().eq('FS369'));test=np.flatnonzero(data.experiment_id.str.upper().eq('FS437'))
        if len(train) and len(test) and data.iloc[train].organoid_id.nunique()>=2:
            for label,(est,cols) in specs.items():
                pred,model=evaluate_split(data,target,train,test,est,cols,label,'FS369_to_FS437','batch',cfg.seed);preds.append(pred)
                if label=='Neural HistGB':saved[target]={'model':model,'columns':cols,'train':data.iloc[train].copy(),'test':data.iloc[test].copy()}
        else:skips.append({'analysis':target+' / batch transfer','reason':'Requires FS369 training cultures and independent FS437 test cultures with compatible labels.'})
        log(f'Models complete: {target}, {n} cultures, {len(data)} eligible windows')
    pred=pd.concat(preds,ignore_index=True) if preds else pd.DataFrame()
    scores,org_scores=metrics_table(pred,cfg)
    pred.to_csv(out/'held_out_predictions.csv',index=False);scores.to_csv(out/'model_comparison.csv',index=False);org_scores.to_csv(out/'per_organoid_errors.csv',index=False)
    atomic_json(out/'skipped_analyses.json',skips)
    return {'predictions':pred,'scores':scores,'organoid_scores':org_scores,'batch_models':saved,'skips':skips}

def run_ablations_and_importance(features,project,results):
    cfg=project['cfg'];out=Path(cfg.output_root);rows=[];importance=[]
    for target,batch in results['batch_models'].items():
        tr,te=batch['train'],batch['test'];est=model_specs(cfg.seed)['Neural HistGB'][0]
        groups={'all_neural':NEURAL_FEATURES}
        if cfg.run_ablations:groups.update({'without_'+k:[x for x in NEURAL_FEATURES if x not in v] for k,v in NEURAL_GROUPS.items()})
        if tr[ENV_FEATURES].notna().any().any() and te[ENV_FEATURES].notna().any().any():
            groups.update(environment_only=ENV_FEATURES,neural_and_environment=NEURAL_FEATURES+ENV_FEATURES)
        for label,cols in groups.items():
            model=clone(est).fit(tr[cols],tr[target]);p=model.predict(te[cols]);mae=pd.DataFrame({'org':te.organoid_id,'e':np.abs(te[target].to_numpy()-p)}).groupby('org').e.mean().mean()
            rows.append({'target':target,'features':label,'FS437_macro_MAE_days':mae,'n_features':len(cols)})
        model=batch['model'];cols=batch['columns']
        pi=permutation_importance(model,te[cols],te[target],scoring='neg_mean_absolute_error',n_repeats=5,random_state=cfg.seed,n_jobs=1)
        importance.extend({'target':target,'feature':c,'MAE_increase_days':m,'repeat_std':s,'evaluation':'FS437 held-out; correlated temporal features limit interpretation'} for c,m,s in zip(cols,pi.importances_mean,pi.importances_std))
        if cfg.run_shap:
            try:
                import shap
                x=model[:-1].transform(te[cols].iloc[:200]);background=model[:-1].transform(tr[cols].iloc[:200])
                values=shap.Explainer(model[-1],background)(x)
                shap.summary_plot(values.values,x,feature_names=cols,show=False)
                plt.savefig(out/f'figure_shap_{target}.png',bbox_inches='tight');plt.close()
            except Exception as e:results['skips'].append({'analysis':'SHAP '+target,'reason':str(e)[:300]+'; permutation importance is still available.'})
    a=pd.DataFrame(rows);i=pd.DataFrame(importance)
    a.to_csv(out/'feature_ablations.csv',index=False);i.to_csv(out/'held_out_feature_importance.csv',index=False)
    return a,i

In [ ]:
# @title Engine 6. Environmental associations, censored survival and external checks
def secondary_analyses(features,project,results):
    cfg=project['cfg'];out=Path(cfg.output_root);status=[]
    d=valid_training_frame(features,'age_days')
    env=[c for c in ENV_FEATURES if len(d) and d[c].notna().sum()>max(10,len(d)//2) and d[c].std()>1e-8]
    if d.organoid_id.nunique()>=5 and len(env):
        try:
            import statsmodels.formula.api as smf
            use=d[['organoid_id','age_days','population_rate_hz']+env].dropna()
            standardized=use.copy()
            for c in ['age_days']+env:standardized[c]=(use[c]-use[c].mean())/use[c].std()
            standardized['log_rate']=np.log1p(standardized.population_rate_hz)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                fit=smf.mixedlm('log_rate ~ age_days + '+' + '.join(env),standardized,groups=standardized.organoid_id).fit(reml=False,method='lbfgs',maxiter=200,disp=False)
            if not fit.converged or caught:raise RuntimeError('Mixed-effects fit is unstable or issued convergence/covariance warnings; coefficients withheld. '+ '; '.join(str(x.message) for x in caught)[:180])
            pd.DataFrame({'term':fit.params.index,'estimate':fit.params.values,'std_error':fit.bse.values,'p_value_descriptive_only':fit.pvalues.values}).to_csv(out/'environment_mixed_effects.csv',index=False)
            status.append({'analysis':'environment associations','status':'completed' if fit.converged else 'nonconverged - do not interpret','note':'Descriptive, not causal. Hourly serial correlation and only two experiments limit inference.'})
        except Exception as e:status.append({'analysis':'environment associations','status':'skipped','note':type(e).__name__+': '+str(e)[:180]})
    else:status.append({'analysis':'environment associations','status':'skipped','note':'Need varying environmental logs and at least five eligible independent cultures.'})
    # One predeclared 24-hour landmark per culture: no repeated-window pseudo-replication.
    landmarks=[]
    for _,r in project['meta'].iterrows():
        if r.role!='primary' or not r.endpoint_verified or pd.isna(r.endpoint_utc):continue
        landmark=r.recording_start_utc.timestamp()+86400
        if r.endpoint_utc.timestamp()<=landmark:continue
        z=features[(features.organoid_id==r.organoid_id)&features.qc_pass&(features.prediction_time_s<=landmark)&(features.prediction_time_s>landmark-cfg.window_seconds)]
        if z.empty:continue
        x=z.sort_values('prediction_time_s').iloc[-1]
        landmarks.append({'organoid_id':r.organoid_id,'followup_days':(r.endpoint_utc.timestamp()-landmark)/86400,
          'event':int(r.endpoint_kind=='confirmed_death'),'rate':x.population_rate_hz,'active_fraction':x.active_fraction})
    survival=pd.DataFrame(landmarks);survival.to_csv(out/'survival_landmark_table.csv',index=False)
    if len(survival)>=20 and survival.event.sum()>=5 and (survival.event==0).sum()>=2:
        try:
            from lifelines import KaplanMeierFitter, CoxPHFitter
            km=KaplanMeierFitter().fit(survival.followup_days,survival.event,label='Verified endpoints: 24-hour landmark')
            ax=km.plot_survival_function();ax.set(xlabel='Days after landmark',ylabel='Survival probability');plt.tight_layout();plt.savefig(out/'figure_survival.png');plt.close()
            km.survival_function_.to_csv(out/'kaplan_meier.csv')
            # Exploratory fit only; out-of-sample survival validation requires more cohorts.
            cph=CoxPHFitter(penalizer=.5).fit(survival.drop(columns='organoid_id'),duration_col='followup_days',event_col='event')
            cph.summary.to_csv(out/'cox_exploratory.csv')
            status.append({'analysis':'landmark survival','status':'exploratory fit','note':'No held-out claim; administrative censoring must be defensible. Separate prospective validation required.'})
        except Exception as e:status.append({'analysis':'landmark survival','status':'skipped','note':str(e)[:200]})
    else:status.append({'analysis':'landmark survival','status':'skipped','note':f'{len(survival)} usable cultures. Require >=20 cultures, >=5 verified deaths, >=2 censored cultures. Unknown recording endings are never labeled deaths.'})
    ext=features[features.role.eq('external')].copy()
    if len(ext):
        ext.to_csv(out/'external_descriptive_features.csv',index=False)
        batch=results['batch_models'].get('age_days')
        if batch:
            n_channels=set(batch['train'].expected_channels)
            comparable=ext.qc_pass & ext.substrate.eq('human_organoid') & ext.expected_channels.isin(n_channels)
            z=ext[comparable].copy()
            if len(z):
                z['predicted_age_days']=batch['model'].predict(z[batch['columns']])
                z['interpretation']='exploratory external prediction; same channel count does not establish compatible detection or sampling'
                z[['organoid_id','experiment_id','age_basis','age_days','predicted_age_days','interpretation']].to_csv(out/'external_human_age_predictions.csv',index=False)
                valid=z.age_basis.eq('biological_creation')&z.age_days.notna()
                status.append({'analysis':'external human age','status':'predictions produced','note':f'{int(valid.sum())} windows have exact comparable age labels. A cohort range alone is not an accuracy test.'})
        # Rat cultures: their own grouped task, never pooled with human biological-age labels.
        rat=ext[ext.qc_pass & ext.substrate.eq('rat_2d_culture') & ext.age_basis.eq('days_since_first_recording') & ext.age_days.notna()].reset_index(drop=True)
        if rat.organoid_id.nunique()>=3:
            pp=[]
            for fold,(tr,te) in enumerate(LeaveOneGroupOut().split(rat,groups=rat.organoid_id),1):
                for label,(est,cols) in model_specs(cfg.seed).items():
                    p,_=evaluate_split(rat,'age_days',tr,te,est,cols,label,'rat_recording_age_only',fold,cfg.seed);pp.append(p)
            pd.concat(pp,ignore_index=True).to_csv(out/'rat_separate_task_predictions.csv',index=False)
            status.append({'analysis':'rat external task','status':'completed','note':'Target is days since first recording; not human biological age and not cross-species intelligence.'})
    else:status.append({'analysis':'external datasets','status':'skipped','note':'No external rows supplied. Primary analysis is still complete; external validation is not claimed.'})
    atomic_json(out/'secondary_analysis_status.json',status)
    return status

In [ ]:
# @title Engine 7. Figures and a self-contained downloadable results report
def savefig(out,name,title,cfg):
    fig=plt.gcf()
    if cfg.mode=='demo':fig.text(.5,.005,'SYNTHETIC DEMONSTRATION - NOT RESEARCH RESULTS',ha='center',color='#a33',fontsize=8)
    if title:fig.suptitle(title,y=1.02)
    fig.tight_layout(rect=(0,.035,1,1));fig.savefig(Path(out)/name,bbox_inches='tight');plt.show();plt.close(fig)

def make_figures(features,project,results,con,ablations,importance):
    cfg=project['cfg'];out=Path(cfg.output_root);f=features
    plt.figure(figsize=(10,4));q=f.groupby('organoid_id').coverage_fraction.mean();q.plot.bar(color=PALETTE[0]);plt.ylim(0,1.05);plt.ylabel('Mean verified coverage fraction');savefig(out,'figure_01_coverage.png','Recording exposure audit',cfg)
    oid=f.organoid_id.iloc[0];r=project['meta'].set_index('organoid_id').loc[oid]
    a=r.recording_start_utc.timestamp();sp=con.execute('SELECT channel,timestamp_s FROM spikes WHERE organoid_id=? AND timestamp_s>=? AND timestamp_s<? ORDER BY timestamp_s LIMIT 100000',[oid,a,a+60]).fetchdf()
    plt.figure(figsize=(10,3))
    for j,c in enumerate(r.channels):
        t=sp.loc[sp.channel==c,'timestamp_s']-a;plt.vlines(t,j,j+.7,color=PALETTE[0],lw=.6)
    plt.yticks(np.arange(len(r.channels))+.3,r.channels);plt.xlabel('Seconds from recording start');plt.ylabel('Declared channel');savefig(out,'figure_02_raster.png',f'{oid}: first 60 seconds (maximum 100,000 displayed spikes)',cfg)
    plt.figure(figsize=(10,4))
    for oid,g in f.groupby('organoid_id'):
        plt.plot(g.elapsed_recording_days,g.population_rate_hz,label=oid,alpha=.75)
    plt.xlabel('Days since recording start');plt.ylabel('Population event rate / second');plt.legend(fontsize=7,ncol=2);savefig(out,'figure_03_activity.png','Observed activity trajectories; gaps remain missing',cfg)
    primary=valid_training_frame(f,'age_days')
    if len(primary):
        fig,axes=plt.subplots(1,2,figsize=(10,3.5));primary.age_days.hist(ax=axes[0],bins=20,color=PALETTE[0]);axes[0].set(xlabel='Biological age / days',ylabel='Eligible windows');primary.groupby('experiment_id').organoid_id.nunique().plot.bar(ax=axes[1],color=PALETTE[1]);axes[1].set(ylabel='Independent cultures');savefig(out,'figure_04_targets.png','Dataset support and target distribution',cfg)
    scores=results['scores'];pred=results['predictions']
    if len(scores):
        age=scores[(scores.target=='age_days')&(scores.scheme=='leave_one_organoid_out')].sort_values('macro_MAE_days')
        if len(age):
            plt.figure(figsize=(10,4));plt.barh(age.model,age.macro_MAE_days,color=PALETTE[0]);plt.xlabel('Mean culture MAE / days (lower is better)');savefig(out,'figure_05_model_comparison.png','Held-out biological-age prediction',cfg)
        z=pred[(pred.target=='age_days')&(pred.model=='Neural HistGB')]
        if len(z):
            fig,axes=plt.subplots(1,2,figsize=(10,4))
            for scheme,g in z.groupby('scheme'):axes[0].scatter(g.true,g.predicted,s=12,alpha=.55,label=scheme)
            lo=min(z.true.min(),z.predicted.min());hi=max(z.true.max(),z.predicted.max());axes[0].plot([lo,hi],[lo,hi],'k--',lw=1);axes[0].set(xlabel='True age / days',ylabel='Predicted age / days');axes[0].legend(fontsize=7)
            zz=z[z.scheme=='leave_one_organoid_out'];axes[1].scatter(zz.true,zz.predicted-zz.true,s=12,alpha=.5);axes[1].axhline(0,color='k',ls='--');axes[1].set(xlabel='True age / days',ylabel='Residual / days');savefig(out,'figure_06_predictions.png','Out-of-sample predictions and errors',cfg)
        rul=pred[(pred.target=='remaining_life_days')&(pred.model=='Neural HistGB')]
        if len(rul):
            plt.figure(figsize=(6,4));plt.scatter(rul.true,rul.predicted,s=15,alpha=.5);plt.xlabel('Confirmed days remaining');plt.ylabel('Predicted days remaining');savefig(out,'figure_07_remaining_life.png','Complete-case verified-death analysis; selection bias remains',cfg)
    if len(ablations):
        z=ablations[ablations.target=='age_days'].sort_values('FS437_macro_MAE_days')
        if len(z):
            plt.figure(figsize=(10,4));plt.barh(z.features,z.FS437_macro_MAE_days,color=PALETTE[0]);plt.xlabel('FS437 mean culture MAE / days');savefig(out,'figure_08_ablations.png','Fixed FS369-to-FS437 feature ablations',cfg)
    if len(importance):
        z=importance[importance.target=='age_days'].sort_values('MAE_increase_days').tail(12)
        if len(z):
            plt.figure(figsize=(9,4));plt.barh(z.feature,z.MAE_increase_days,xerr=z.repeat_std,color=PALETTE[0]);plt.xlabel('MAE increase after shuffling / days');savefig(out,'figure_09_importance.png','Held-out permutation importance (descriptive)',cfg)
    end=f[f.qc_pass & f.remaining_life_days.notna()].copy()
    if len(end):
        correlations=[];plt.figure(figsize=(10,4))
        for oid,g in end.groupby('organoid_id'):
            g=g.sort_values('remaining_life_days');median=g.population_rate_hz.median()
            scale=median if median>0 else 1.
            plt.plot(g.remaining_life_days,g.population_rate_hz/scale,label=oid,alpha=.7)
            correlations.append({'organoid_id':oid,'n_windows':len(g),'spearman_rate_vs_remaining':g.population_rate_hz.corr(g.remaining_life_days,method='spearman') if g.population_rate_hz.nunique()>1 and len(g)>=5 else np.nan,'interpretation':'descriptive association only; no independent-window p-value'})
        plt.gca().invert_xaxis();plt.xlabel('Verified days remaining (toward endpoint →)');plt.ylabel('Population rate / within-culture median');plt.legend(fontsize=7,ncol=2)
        savefig(out,'figure_10_endpoint_alignment.png','Activity aligned to verified endpoints; descriptive normalization only',cfg)
        pd.DataFrame(correlations).to_csv(out/'endpoint_activity_associations.csv',index=False)

def export_report(features,project,results,secondary,ablations):
    cfg=project['cfg'];out=Path(cfg.output_root)
    atomic_json(out/'run_configuration.json',asdict(cfg))
    packages={n:importlib.metadata.version(n) for n in ['numpy','pandas','scipy','scikit-learn','h5py','pyarrow','duckdb','matplotlib']}
    atomic_json(out/'environment_versions.json',{'python':sys.version,'packages':packages,'engine':VERSION,'utc_run':datetime.now(timezone.utc).isoformat()})
    status=json.loads((out/'analysis_status.json').read_text())
    warnings_list=[
      'This report is private by default. Licensed recordings, derived features, figures and results may have sharing restrictions. Nothing was published.',
      'Biological age requires a verified creation date. Days since first recording is a different target.',
      'Remaining-life regression uses only verified deaths and can suffer selection bias. Unknown endings are not death labels.',
      'All model features are from the preceding window; labels use the prediction time at its end. No culture identifiers, endpoint timestamps or future logs are predictors.',
      'Culture-level bootstrap intervals describe this small sample, not a population guarantee. Neighboring windows and only two experiments limit generalization.',
      'The clock model is an explicitly non-neural diagnostic; a strong clock baseline may show that maturation is confounded with recording time.',
      'Neither spontaneous activity, age prediction nor network bursts establishes learning, intelligence or consciousness.',
      'A cohort age range is a plausibility check, not external accuracy validation. Rat recording-age models are separate tasks.',
      'Feature ablations and importance are explanatory analyses on the held-out batch. Do not tune and then re-claim FS437 as untouched confirmation.',
      'Amplitude units are source-defined; waveform features require aligned event snippets. Raw voltage is not automatically spike-sorted.',
    ]
    if cfg.mode=='demo':warnings_list.insert(0,'SYNTHETIC DEMONSTRATION ONLY. These numbers cannot be used as BDAPP research results.')
    body=['<h1>BDAPP: organoid age and remaining life</h1>',f'<p>Mode: <b>{html.escape(cfg.mode.upper())}</b> | Engine {VERSION} | {datetime.now(timezone.utc).isoformat()}</p>',
      '<h2>Data audit</h2>',pd.DataFrame([status]).to_html(index=False,escape=True),'<h2>Held-out comparisons</h2>',
      results['scores'].to_html(index=False,float_format=lambda x:f'{x:.4f}') if len(results['scores']) else '<p>No eligible supervised task. See the quality audit and skipped analyses.</p>',
      '<h2>Interpretation and limits</h2>','<ul>'+''.join('<li>'+html.escape(x)+'</li>' for x in warnings_list)+'</ul>',
      '<h2>Skipped or conditional analyses</h2>',pd.DataFrame(results['skips']+secondary).to_html(index=False,escape=True),'<h2>Figures</h2>']
    for p in sorted(out.glob('figure_*.png')):
        body.append('<h3>'+html.escape(p.stem)+'</h3><img src="data:image/png;base64,'+base64.b64encode(p.read_bytes()).decode()+'">')
    body.append('<h2>Methods and sources</h2><p>Hourly windows; verified exposure; fixed-ISI channel bursts (100 ms, at least 5 spikes); network events use >=75% of channels in fixed 100 ms bins. Timing/bin features are missing for partially observed windows. Models use fixed hyperparameters and training-fold-only preprocessing.</p>')
    body.append('<p>FinalSpark platform: <a href="https://doi.org/10.3389/frai.2024.1376042">Jordan et al. 2024</a>. Public external records: <a href="https://zenodo.org/records/22755004">TU Munich 22755004</a>; <a href="https://dandiarchive.org/dandiset/002020">DANDI 002020</a>. Grouped validation: <a href="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html">scikit-learn</a>. Source permissions must be checked independently.</p>')
    text='<!doctype html><html><head><meta charset="utf-8"><title>BDAPP private analysis report</title><style>body{font:15px/1.6 system-ui;max-width:1120px;margin:40px auto;color:#06110d;padding:20px;background:#f6f5ef}h1,h2{color:#064c37}table{border-collapse:collapse;display:block;overflow:auto;font-size:12px}th,td{border:1px solid #ccd4ce;padding:7px}th{background:#e2ebe4}img{max-width:100%;background:white}li{margin-bottom:7px}@media print{img{max-height:650px}h2{break-after:avoid}}</style></head><body>'+''.join(body)+'</body></html>'
    (out/'BDAPP_REPORT.html').write_text(text,encoding='utf8')
    (out/'INTERPRETATION.txt').write_text('\n'.join(warnings_list),encoding='utf8')
    atomic_json(out/'skipped_analyses.json',results['skips'])
    archive=out/'BDAPP_PRIVATE_RESULTS.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in out.iterdir():
            if p.is_file() and p!=archive and p.suffix.lower() in {'.csv','.json','.html','.txt','.png','.parquet'}:z.write(p,p.name)
    log(f'Private report ready: {out / "BDAPP_REPORT.html"}')
    return archive

In [ ]:
# @title Engine 8. Native public-data readers (separate descriptive track)
def matlab_text(dataset):
    return ''.join(chr(int(x)) for x in np.asarray(dataset).reshape(-1) if int(x))

def read_munich_mat(path):
    """Verified against a real file in Zenodo 22755004, MATLAB v7.3 export."""
    with h5py.File(path,'r') as f:
        if not {'spike_times','metadata'}<=set(f):raise ValueError('Not the supported TU Munich organoid MAT schema.')
        meta=f['metadata'];duration=float(np.asarray(meta['recording_duration']).ravel()[0])
        channels=np.asarray(meta['channel_ids']).ravel().astype(int).astype(str).tolist()
        refs=np.asarray(f['spike_times']).ravel()
        oid=matlab_text(f['organoid_id']) if 'organoid_id' in f else Path(path).stem
        records=[]
        amps=np.asarray(f['amplitudes']).ravel() if 'amplitudes' in f else None
        if len(refs)!=len(channels):raise ValueError('MAT spike-time cells do not match channel metadata.')
        for i,(c,ref) in enumerate(zip(channels,refs)):
            obj=f[ref]
            t=np.asarray(obj,dtype=float).ravel() if not obj.attrs.get('MATLAB_empty',False) else np.array([])
            t=t[np.isfinite(t)]
            av=np.asarray(f[amps[i]],dtype=float).ravel() if amps is not None else np.full(len(t),np.nan)
            if len(av)!=len(t):av=np.full(len(t),np.nan)
            records.append(pd.DataFrame({'channel':c,'timestamp_s':t,'amplitude':av,'waveform_p2p':np.nan,'waveform_width_us':np.nan}))
        a=pd.concat(records,ignore_index=True)
        if duration<=0 or (len(a) and ((a.timestamp_s<0).any() or (a.timestamp_s>=duration+1e-6).any())):raise ValueError('MAT spike times contradict recording_duration; do not guess another time unit.')
        return oid,channels,duration,a

def read_nwb_units(path):
    """NWB Units spike trains only; unit identity is not electrode identity."""
    with h5py.File(path,'r') as f:
        if 'units/spike_times' not in f or 'units/spike_times_index' not in f:raise ValueError('NWB has no Units spike_times table. Raw electrical series need explicit spike sorting or a source-specific adapter.')
        times=f['units/spike_times'];ends=np.asarray(f['units/spike_times_index']).astype(int)
        ids=np.asarray(f['units/id']).astype(str) if 'units/id' in f else np.arange(len(ends)).astype(str)
        if len(ends)>4096 or len(times)>2_000_000:raise ValueError('NWB descriptive size guard: use selected spike-sorted assets or an explicit chunked export for this file.')
        # Prefer acquisition timestamps for exposure; otherwise do NOT use last spike as recording duration.
        duration=None
        if 'intervals/epochs/start_time' in f and 'intervals/epochs/stop_time' in f:
            starts=np.asarray(f['intervals/epochs/start_time']);stops=np.asarray(f['intervals/epochs/stop_time'])
            if len(starts)==1 and len(stops)==1 and abs(starts[0])<1e-6:duration=float(stops[0])
        records=[];lo=0
        for uid,hi in zip(ids,ends):
            t=np.asarray(times[lo:hi],dtype=float);lo=hi
            records.append(pd.DataFrame({'channel':str(uid),'timestamp_s':t,'amplitude':np.nan,'waveform_p2p':np.nan,'waveform_width_us':np.nan}))
        return Path(path).stem,ids.tolist(),duration,pd.concat(records,ignore_index=True)

def run_public_descriptive(cfg):
    root=Path(cfg.data_root);out=Path(cfg.output_root);rows=[];issues=[]
    # An explicit exposure table can supply NWB duration without guessing from firing.
    ep=root/'external_recordings.csv'
    extra=pd.read_csv(ep).fillna('') if ep.exists() else pd.DataFrame(columns=['path','duration_s'])
    durations={str(a):float(b) for a,b in zip(extra.path,extra.duration_s)}
    for path in sorted(root.rglob('*')):
        if path.suffix.lower() not in {'.mat','.nwb'}:continue
        try:
            oid,channels,duration,a=read_munich_mat(path) if path.suffix.lower()=='.mat' else read_nwb_units(path)
            if duration is None:duration=durations.get(str(path.relative_to(root)))
            if duration is None:
                rows.append({'recording_id':oid,'source':str(path.relative_to(root)),'channel_kind':'sorted_unit','n_channels':len(channels),'n_spikes':len(a),'duration_s':np.nan,'note':'Exposure unavailable; no rates or age claims computed. Add external_recordings.csv with verified duration_s.'});continue
            if duration<=0:raise ValueError('Verified duration must be positive.')
            if len(a) and ((a.timestamp_s<0).any() or (a.timestamp_s>=duration).any()):raise ValueError('Spike times exceed declared exposure duration.')
            feat=neural_features(a,channels,0,duration,duration)
            rows.append({'recording_id':oid,'source':str(path.relative_to(root)),'channel_kind':'electrode' if path.suffix.lower()=='.mat' else 'sorted_unit','n_channels':len(channels),'n_spikes':len(a),'duration_s':duration,**feat})
        except Exception as e:issues.append({'path':str(path.relative_to(root)),'reason':str(e)[:300]})
    df=pd.DataFrame(rows);df.to_csv(out/'public_recording_features.csv',index=False);atomic_json(out/'public_adapter_issues.json',issues)
    if len(df) and 'network_burst_rate_min' in df:
        plt.figure(figsize=(9,4));z=df.dropna(subset=['network_burst_rate_min']);plt.hist(z.network_burst_rate_min,bins=15,color=PALETTE[0]);plt.xlabel('Network burst events / minute');plt.ylabel('Recordings');savefig(out,'figure_public_bursts.png','External descriptive activity; no learning or age-accuracy inference',cfg)
    log(f'External descriptive adapter: {len(rows)} recordings, {len(issues)} unsupported/flagged files')
    return df,issues

In [ ]:
# @title Engine 9. Explicit synthetic demonstration; never a fallback for missing real files
def create_demo_dataset(root,seed=42):
    root=Path(root);root.mkdir(parents=True,exist_ok=True);rng=np.random.default_rng(seed)
    meta=[];frames={};cmap=[];base=pd.Timestamp('2026-01-01',tz='UTC');hours=12
    for experiment in ['FS369','FS437']:
        parts=[]
        for k in range(4):
            oid=f'{experiment}-org{k}';ch=[str(x) for x in range(k*8,k*8+8)]
            age=30+15*k+(4 if experiment=='FS437' else 0)
            m={c:'' for c in META_COLS};m.update(organoid_id=oid,experiment_id=experiment,role='primary',substrate='human_organoid',creation_utc=str(base-pd.Timedelta(days=age)),age_basis='biological_creation',recording_start_utc=str(base),recording_end_utc=str(base+pd.Timedelta(hours=hours)),coverage_verified=True,channel_ids=';'.join(ch),endpoint_utc=str(base+pd.Timedelta(days=2+k)),endpoint_kind='confirmed_death',endpoint_verified=True,endpoint_evidence='SYNTHETIC FIXTURE, NOT A REAL DEATH')
            if experiment=='FS437' and k==3:m.update(endpoint_kind='administrative_censoring')
            if experiment=='FS369' and k==3:m.update(endpoint_verified=False,endpoint_kind='unknown',endpoint_evidence='')
            meta.append(m)
            cmap.extend({'experiment_id':experiment,'channel':c,'organoid_id':oid} for c in ch)
            for h in range(hours):
                for c in ch:
                    n=int(rng.poisson(20+age*.8+h))
                    t=base.timestamp()+h*3600+rng.uniform(0,3599.99,n)
                    parts.append(pd.DataFrame({'Time':t,'channel':int(c),'Max Amplitude':rng.normal(-35,4,n)}))
        frames[experiment]=pd.concat(parts,ignore_index=True).sample(frac=1,random_state=seed).reset_index(drop=True)
    frames['FS369'].to_csv(root/'fs369_spikes.csv',index=False)
    data=frames['FS437'];dt=np.dtype([('Time','f8'),('channel','i4'),('Max Amplitude','f4')]);a=np.empty(len(data),dtype=dt)
    for c in data:a[c]=data[c].to_numpy()
    with h5py.File(root/'fs437_spikes.h5','w') as f:f.create_dataset('spikes',data=a,chunks=True,compression='gzip')
    pd.DataFrame(meta).to_csv(root/'organoids.csv',index=False);pd.DataFrame(cmap).to_csv(root/'channel_map.csv',index=False)
    env=[]
    for exp in frames:
        for h in range(hours):env.append({'experiment_id':exp,'timestamp_utc':str(base+pd.Timedelta(hours=h,minutes=15)),'temperature_c':37+.1*np.sin(h),'co2_pct':5.,'humidity_pct':90.})
    pd.DataFrame(env).to_csv(root/'environment.csv',index=False)
    pd.DataFrame([{'experiment_id':'FS369','organoid_id':'','timestamp_utc':str(base+pd.Timedelta(hours=4,minutes=10)),'event_type':'medium_change','exclude_before_min':10,'exclude_after_min':20}]).to_csv(root/'events.csv',index=False)
    (root/'SYNTHETIC_DATA_ONLY.txt').write_text('Generated demonstration data. Never use these model scores as research findings.')
    return root

## 4 · Inspect the files and prepare setup templates
Inspection reads small samples and HDF5 schemas. The inventory distinguishes recognized event tables from raw snippets and unsupported layouts. Raw voltage matrices are not spike events and are not automatically spike-sorted.

**Supported primary exports**
- CSV / TSV / Parquet with a timestamp and channel per event.
- HDF5 with a one-dimensional compound event table, or aligned one-dimensional timestamp/channel arrays in a group.
- An optional aligned waveform array can provide amplitude/width features. Set `waveform_key` and `sampling_rate_hz` in `sources.csv`.

ZIPs should already be extracted. Source files remain unchanged. After an interruption, rerun with the same folders; completed normalized sources and feature partitions are reused.

In [ ]:
from pathlib import Path
if MODE == 'demo':
    DATA_ROOT = str(Path('/content/BDAPP_DEMO_data') if Path('/content').exists() else Path.cwd()/'BDAPP_DEMO_data')
    OUTPUT_ROOT = str(Path(OUTPUT_ROOT)/'synthetic_demo')
    create_demo_dataset(DATA_ROOT,SEED)
cfg=Config(DATA_ROOT,OUTPUT_ROOT,mode=MODE,window_seconds=WINDOW_SECONDS,
           min_coverage=MIN_COVERAGE,chunk_rows=CHUNK_ROWS,max_window_events=MAX_WINDOW_EVENTS,
           source_timezone=SOURCE_TIMEZONE,seed=SEED,run_ablations=RUN_ABLATIONS,run_shap=RUN_SHAP)
initialize_output(cfg)
inventory,suggested_sources=inspect_files(cfg)
print(f'{len(inventory):,} candidate files; {len(suggested_sources):,} recognized event sources')
display(inventory.head(100))
display(suggested_sources)
template_columns={
 'organoids_TEMPLATE.csv':META_COLS,
 'channel_map_TEMPLATE.csv':['experiment_id','channel','organoid_id'],
 'coverage_TEMPLATE.csv':['organoid_id','start_utc','end_utc'],
 'events_TEMPLATE.csv':['experiment_id','organoid_id','timestamp_utc','event_type','exclude_before_min','exclude_after_min'],
 'environment_TEMPLATE.csv':['experiment_id','timestamp_utc','temperature_c','co2_pct','humidity_pct','flow_rate'],
 'external_recordings_TEMPLATE.csv':['path','duration_s'],
}
for name,cols in template_columns.items():
    p=Path(OUTPUT_ROOT)/name
    if not p.exists(): pd.DataFrame(columns=cols).to_csv(p,index=False)
suggested_sources.to_csv(Path(OUTPUT_ROOT)/'sources_TEMPLATE.csv',index=False)
print('Inventory and setup templates are in OUTPUT_ROOT. They contain no invented dates or labels.')

## 5 · One-time metadata setup
Place `organoids.csv` directly in `DATA_ROOT`. Use one row per independent living culture. The identifier must be globally unique across experiments; a chip name alone is not an organoid ID.

| Column | What to supply |
|---|---|
| `organoid_id` | Your exact culture ID, used consistently in sources/channel mapping. |
| `experiment_id` | `FS369` or `FS437` for the primary batch-transfer test. |
| `role` | `primary` for the FS task; `external` for separately mapped external files. |
| `substrate` | `human_organoid` or `rat_2d_culture`. |
| `creation_utc` | Verified neurosphere creation timestamp. Leave blank if unknown. |
| `age_basis` | `biological_creation`; use `days_since_first_recording` only for a separate recording-age task. |
| `recording_start_utc`, `recording_end_utc` | Actual recording bounds. End is exclusive. Use ISO dates with timezone, such as `2026-01-01T12:00:00Z`; replace this illustrative date with the real one. |
| `coverage_verified` | `True` only if recording was continuously observed across these bounds. Otherwise supply real intervals in `coverage.csv`; missing exposure stays unknown. |
| `channel_ids` | The exact channels belonging to this organoid, separated by semicolons. Verify whether the source uses zero/one indexing. |
| `endpoint_utc` | Verified death or administrative censoring timestamp, if known. |
| `endpoint_kind` | `confirmed_death`, `administrative_censoring`, or `unknown`. |
| `endpoint_verified` | `True` only with independent evidence for the specified endpoint. |
| `endpoint_evidence` | A short source/reference, not an assumption that the last spike means death. |

**If a chip source mixes four organoids:** add `channel_map.csv` with `experiment_id,channel,organoid_id`, unless every event already includes a correct global organoid ID. The reader does not guess four groups of eight.

**If source detection needs correction:** copy `sources_TEMPLATE.csv` to `DATA_ROOT/sources.csv`, then edit its mapping. `time_kind` accepts `iso`, `unix_s`, `unix_ms`, `unix_us`, `unix_ns`, `relative_s`, `relative_ms`, `relative_us`, `relative_ns`, or `samples`. Relative times require the real `origin_utc`; sample indices also require `sampling_rate_hz`. `hdf_key` identifies the event dataset/group.

**Optional logs**
- `coverage.csv`: `organoid_id,start_utc,end_utc`; known observed intervals, merged without double counting overlaps.
- `events.csv`: event time/type plus explicit before/after exclusion minutes. For chip-wide events, leave `organoid_id` blank.
- `environment.csv`: timestamp, experiment and available temperature/CO₂/humidity/flow columns. Missing logs are reported; no environmental conclusion is invented.

This setup is necessary because birth dates, deaths, channel assignments and exposure cannot be inferred reliably from spike activity. Once saved, you reuse these files on later runs.

## 6 · Public datasets — separate descriptive check
The native TU Munich reader was checked against actual MATLAB v7.3 files from the archive. It reads each file's `metadata/recording_duration`, channel IDs, spikes and amplitudes. No per-organoid age is inferred from the cohort range.

NWB support here reads a **Units spike-times table**, not raw ElectricalSeries. Sorted units are labeled as units, not electrodes. If duration is not explicitly available, add `external_recordings.csv` with relative file path and verified `duration_s`; duration is never taken from the last spike. Large/unsupported files are listed in `public_adapter_issues.json`.

These short recordings produce descriptive features. They are not silently pooled into the primary hourly prediction task. For a separate rat recording-age model, supply explicit compatible event exports and metadata with `role=external`, `substrate=rat_2d_culture`, and the correctly labeled recording-age origin.

In [ ]:
public_features,public_issues=run_public_descriptive(cfg)
if len(public_features): display(public_features.head(20))
if public_issues: display(pd.DataFrame(public_issues).head(30))

## 7 · Validate metadata and ingest the primary data
Missing metadata or ambiguous units stop the supervised pipeline with a specific instruction. Nothing substitutes demo data. Numeric time auto-detection is restricted to plausible absolute Unix timestamps; relative units require your mapping.

The source cache is keyed by file size, modification time, first/last-block digest, schema and metadata. This is a restart cache, not a full cryptographic integrity audit. Normalized event storage needs free space; the report ZIP excludes this raw-event cache.

In [ ]:
project=prepare_project(cfg,suggested_sources)
display(project['meta'].drop(columns=['channels']))
connection,normalized_paths,ingestion_key=ingest(project)
print('Ingestion complete:',len(normalized_paths),'source cache files')

## 8 · Extract hourly neural features
**Forecast time is the end of the preceding window.** Features never use events after that point. Rates use verified exposure. Fully observed zero-spike windows are genuine zero rates; unknown coverage is not silence. Duplicate event keys `(organoid,channel,timestamp)` across shards are collapsed and counted in the audit.

| Feature group | Definitions |
|---|---|
| Rates | Population and channel rate, active-channel fraction, channel-rate dispersion/Gini. |
| Timing | ISI median, CV, CV2, fraction below 10 ms. |
| Bursts | Fixed-ISI channel bursts: adjacent ISI ≤100 ms, ≥5 spikes. Network events: ≥75% of channels in fixed 100 ms bins; adjacent bins merged. These definitions differ from logISI/sliding-window methods. |
| Population | One-second Fano factor and count entropy; pairwise binned correlation; fraction of occupied 100 ms bins with ≥2 active channels. |
| Amplitude | Source-unit absolute amplitude median; optional aligned waveform peak-to-peak and max/min separation in µs. |

These are activity summaries, not direct measures of intelligence or consciousness. A low firing rate alone does not establish tissue death. Partially observed windows retain rates but have missing timing/binned features.

In [ ]:
features=extract_features(project,connection,ingestion_key)
display(features.groupby(['experiment_id','role']).agg(windows=('organoid_id','size'),cultures=('organoid_id','nunique'),eligible_windows=('qc_pass','sum')))
display(features.head())
print('Features saved:',Path(OUTPUT_ROOT)/'hourly_features.parquet')
if not features.qc_pass.any():
    print('No eligible windows: inspect window_quality_audit.csv and coverage metadata. Supervised results will be explicitly skipped.')

## 9 · Held-out prediction and baselines
- **Primary outcome:** equal-weighted mean MAE across held-out cultures, in days. Pooled MAE/RMSE/R² are supplemental.
- **Leave one organoid out:** all hours of the held-out culture remain outside fitting and preprocessing.
- **Batch transfer:** fixed models train on FS369 and test on FS437. Hyperparameters are fixed; no test-batch tuning occurs.
- **Models:** median, firing-rate-only Ridge, all-neural Ridge, histogram gradient boosting, a non-neural clock diagnostic, and a whole-culture target-curve shuffle control. The shuffle control is a sanity check, not a permutation p-value.
- **Predictor whitelist:** only declared activity features. No culture IDs, absolute timestamps, birth/death times or days-left labels are neural predictors. The clock diagnostic is separately labeled.
- **Uncertainty:** cultures, not individual hours, are bootstrapped. With fewer than three test cultures, no bootstrap interval is reported. A small cohort still limits inference.
- **Remaining life:** evaluated only for verified deaths. Complete-case selection bias is explicit; unknown endings never become death events.

At least three independent eligible cultures are required for a supervised task. A target that lacks reliable labels is reported as skipped, while other tasks can proceed.

In [ ]:
results=run_models(features,project)
display(results['scores'])
if results['skips']:display(pd.DataFrame(results['skips']))

## 10 · Ablation and explanation
Group ablations remove rates, timing, bursts, population or amplitude features in turn. Where usable environmental logs exist, environment-only and combined models are also compared on FS437. This tests predictive contribution; it does not identify causal maturation effects.

Permutation importance is evaluated on held-out FS437. Correlated/temporal features limit its interpretation. SHAP is optional; if desired, install `shap` in an additional cell and set `RUN_SHAP=True`. If unavailable or incompatible, the report records the reason and retains permutation importance. Do not select a model from these held-out results and then describe FS437 as untouched confirmation.

In [ ]:
ablations,importance=run_ablations_and_importance(features,project,results)
display(ablations)
if len(importance):display(importance.sort_values('MAE_increase_days',ascending=False).head(15))

## 11 · Environmental and endpoint-aware extensions
The optional mixed-effects analysis relates log activity to age and available environmental measurements with a culture intercept. Unstable fits have their coefficients withheld. Serial dependence and two experiments prevent causal conclusions.

Survival analysis uses **one 24-hour landmark per culture**. It requires verified death/censoring, at least 20 cultures, five deaths and two censored observations. This is an exploratory fit, not an out-of-sample survival claim. Your two-experiment dataset may not meet these criteria; a clear skip is a valid result. The notebook does not claim that 300 GB implies many independent samples.

In [ ]:
secondary_status=secondary_analyses(features,project,results)
display(pd.DataFrame(secondary_status))

## 12 · Observe the figures
Figures cover verified exposure, an actual 60-second raster, activity trajectories, target support, model comparison, held-out predictions/residuals, eligible remaining-life results, feature ablations and importance. A figure is produced only when the corresponding data/analysis exists.

In [ ]:
make_figures(features,project,results,connection,ablations,importance)

## 13 · Final report and download
The self-contained HTML report embeds its figures and opens offline. It includes data coverage, model scores, skipped analyses, methods, limitations and source links. It can be printed to PDF from your browser. The ZIP contains derived results, **not the original recordings or normalized raw-event cache**; derived material can still be restricted by your agreement.

Main files: `BDAPP_REPORT.html`, `model_comparison.csv`, `held_out_predictions.csv`, `per_organoid_errors.csv`, `hourly_features.parquet`, `feature_ablations.csv`, `held_out_feature_importance.csv`, quality/mapping audits, figures, `environment_versions.json`, and explicit analysis-status JSON files.

No automatic upload, public sharing, email or live-platform command occurs.

In [ ]:
private_results_zip=export_report(features,project,results,secondary_status,ablations)
display(HTML('<h3>Analysis finished</h3><p>Open <b>BDAPP_REPORT.html</b> in the configured results folder. Read the audit and skipped analyses before interpreting model scores.</p>'))
print('Report:',Path(OUTPUT_ROOT)/'BDAPP_REPORT.html')
print('Results ZIP:',private_results_zip)
connection.close()

In [ ]:
DOWNLOAD_ZIP = False # @param {type:"boolean"}
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(str(private_results_zip))
    except ImportError:
        print('Outside Colab; open the ZIP at:',private_results_zip)
else:
    print('Results remain in your output folder. Set DOWNLOAD_ZIP=True and run this cell to download them.')

## How this supports your final presentation
1. **Data:** independent cultures and experiments, observed hours, missing coverage, usable labels and exclusions. Report actual processed sizes from the inventory, not a guessed 300 GB.
2. **Methods:** out-of-core event processing, fixed feature definitions, fold-isolated fitting, baselines and grouped testing.
3. **Results:** held-out MAE by model and culture; FS369 → FS437 transfer; explain gaps and negative results.
4. **Ablation:** whether richer features add value beyond firing rate or the clock diagnostic; predictive environmental contribution.
5. **Limitations:** few independent samples, endpoint uncertainty, serial correlation, hardware differences, external-label limits and licensed use.
6. **Future work:** additional independent batches and prospective endpoint verification. These analyses alone do not show a computational or energy advantage over AI.

## Sources and technical references
- Your supplied 12-slide **BDAPP proposal**, especially research questions, method and evaluation slides.
- Jordan et al. (2024), [FinalSpark Neuroplatform](https://doi.org/10.3389/frai.2024.1376042).
- Itatani and Zavaglia, [Zenodo 22755004](https://zenodo.org/records/22755004): public organoid activity; file-specific exposure used by the native reader.
- Nakamuta et al., [DANDI 002020](https://dandiarchive.org/dandiset/002020): separate rat-culture data, not automatically biological-age ground truth.
- [LeaveOneGroupOut documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html).
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html): runtime/resource and Drive I/O limitations.
- [FinalSpark SDK documentation](https://finalspark-np.github.io/np-docs/np_core/doc_v2.html): context for event columns; the notebook uses exported files, not live hardware.

**Validation note:** shipped notebook outputs are intentionally empty. Synthetic tests validate software behavior, not biological hypotheses. Results only become yours after your actual data and verified metadata are processed.